# Getting Started with Claude Opus 4.8 on Amazon Bedrock

**Anthropic's most intelligent Opus model — stronger across coding, agentic tasks, and professional work, with the consistency and autonomy to handle long-running work.**

Claude Opus 4.8 is a meaningful step forward from Opus 4.7, with improvements across the workflows teams run in production. Opus 4.8 improves agentic coding, deepens knowledge work, and increases performance for tasks that span longer horizons.

---

## What's new in Claude Opus 4.8

- **Best coding model available**: State-of-the-art on SWE-bench Pro, SWE-bench Verified, and Terminal-Bench 2.0. Reads codebases like an engineer, plans before it edits, and holds context across long sessions.
- **Agentic step change**: Creative problem-solving and reliable tool use across multi-step workflows. Finds paths around obstacles, recovers from errors, knows when to ask for help vs. keep going.
- **Long-horizon autonomy**: Works independently for longer than any previous model. Plans across stages, tracks dependencies, and re-scopes around problems rather than waiting for human intervention.
- **Enterprise consistency**: Lower output variance, stronger instruction-following. Produces outputs teams can ship without re-checking.
- **Deeper knowledge work**: Synthesizes across sources, makes sensible assumptions on underspecified requests, and self-checks output to get things right on the first pass.

---

## When to use Claude Opus 4.8

| Use case | Description |
|----------|-------------|
| **Agentic coding** | Feature work, large-scale migrations, refactors, and debugging sustained across long sessions in real codebases |
| **Research & analysis** | Synthesizing long, complex sources into briefs, analyses, and structured deliverables |
| **Agents & automation** | Customer-facing and internal agents that use tools reliably and run multi-step workflows with limited oversight |
| **Financial services** | Investment research, earnings analysis, credit/risk review, compliance workflows |
| **Legal** | Contract review, due diligence, case research, first drafts of motions and memos |
| **Life sciences** | Literature review, clinical documentation, regulatory submission drafting |
| **Cybersecurity** | Threat intelligence, vulnerability finding, alert triage, incident response, security code review |

---

## Key capabilities

| Capability | Details |
|------------|---------|
| **Context window** | Up to 1M tokens |
| **Max output** | Up to 128K tokens |
| **Adaptive thinking** | Extended reasoning that activates when needed, with configurable effort levels |
| **Task budgets** | Token spending limits for agentic loops (beta) |
| **Context compaction** | Automatic summarization for infinite context (beta) |
| **Long-horizon autonomy** | Tracks dependencies, recovers from errors, works independently across multi-stage projects |
| **Vision** | High-resolution image input for screenshots, diagrams, charts, and design files |

---

## API options on Amazon Bedrock

Claude Opus 4.8 is available through **Amazon Bedrock Mantle**:

| API | Endpoint | When to use |
|-----|----------|-------------|
| **Messages API** |  | Recommended. Full feature support via Anthropic's native format. |
| **InvokeModel API** |  | Bedrock native API with Anthropic request body. Supports compaction. |
| **Converse API** |  | Bedrock unified conversational API. Simplest integration. |

The **Messages API via Bedrock Mantle** is the recommended starting point for new integrations.

---

## 1. Setup and configuration

In [ ]:
# Install required packages (uncomment if needed)
#!pip install boto3 --upgrade 2>&1 | tail -3 
#!pip install "anthropic[bedrock]" --upgrade 2>&1 | tail -3 

In [ ]:
import boto3
import json
import base64
from botocore.config import Config

# Try importing Anthropic Bedrock Mantle client
try:
    from anthropic import AnthropicBedrockMantle
    MANTLE_AVAILABLE = True
except ImportError:
    print(
        "Note: anthropic[bedrock] not installed. "
        "Messages API examples will be skipped. "
        "Install with: pip install anthropic[bedrock]"
    )
    MANTLE_AVAILABLE = False

# Configuration
REGION = "us-east-1"

# Mantle (Messages API) - recommended
MANTLE_MODEL_ID = "anthropic.claude-opus-4-8"
MANTLE_BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws"

# Runtime (Converse / InvokeModel APIs)
RUNTIME_MODEL_ID = "us.anthropic.claude-opus-4-8"

# Longer timeout for thinking/compaction responses
FAST_CONFIG = Config(read_timeout=1200, retries={"max_attempts": 1})

# Initialize clients
session = boto3.Session()
bedrock_runtime = session.client(
    service_name="bedrock-runtime",
    region_name=REGION,
    config=FAST_CONFIG,
)

if MANTLE_AVAILABLE:
    mantle_client = AnthropicBedrockMantle(
        aws_region=REGION,
        base_url=f"{MANTLE_BASE_URL}/anthropic",
    )

print(f"Region:          {REGION}")
print(f"Mantle Model:    {MANTLE_MODEL_ID}")
print(f"Runtime Model:   {RUNTIME_MODEL_ID}")
print(f"Mantle Base URL: {MANTLE_BASE_URL}")
print(f"Mantle SDK:      {'available' if MANTLE_AVAILABLE else 'not installed'}"  )
print("Setup complete.")

---

## 2. Basic usage with Messages API (Bedrock Mantle)

The Messages API is the **recommended way** to use Claude Opus 4.8 on Amazon Bedrock. It uses Anthropic's native request/response format, authenticated with AWS SigV4 via the `anthropic[bedrock]` SDK.

In [ ]:
# Messages API - basic request
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=2048,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "What are three key considerations when designing "
                        "a distributed system for high availability?"
                    ),
                }
            ],
        )

        print(message.content[0].text)
        print(
            f"\nUsage: Input={message.usage.input_tokens}, "
            f"Output={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock] for Messages API support")

---

## 3. Adaptive thinking

Adaptive thinking gives Claude the freedom to reason deeply when a task requires it, and respond quickly when it doesn't. Instead of fixed token budgets, you guide reasoning depth with an `effort` parameter.

**How it works:**
- Set `thinking.type` to `"adaptive"` — Claude decides whether to think based on task complexity
- Control depth with `output_config.effort` — higher effort = deeper reasoning on complex tasks
- Claude 4+ models use **summarized thinking** by default: the thinking block contains a cryptographic `signature` proving reasoning occurred, but the raw thinking text is not exposed

**Important:** Do NOT use `thinking.type: "enabled"` — it is not supported. Always use `"adaptive"`.

### Effort levels

| Effort | Behavior | Best for |
|--------|----------|----------|
| **max** | Maximum reasoning depth | Most complex analytical workloads |
| **high** | Deep reasoning (default) | Complex tasks — start here |
| **medium** | Balanced — may skip thinking for simple queries | Mixed workloads |
| **low** | Minimal thinking, prioritizes speed | Simple queries, cost-sensitive |

In [ ]:
# Adaptive thinking - basic example
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=16000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": "high"}},
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Design a fault-tolerant distributed consensus algorithm "
                        "that handles Byzantine failures in an asynchronous "
                        "network with up to 1/3 faulty nodes."
                    ),
                }
            ],
        )

        # Check if thinking was triggered
        has_thinking = any(block.type == "thinking" for block in message.content)
        print(f"Claude decided to think: {has_thinking}\n")

        for block in message.content:
            if block.type == "thinking":
                sig_len = len(block.signature) if block.signature else 0
                print(f"[Thinking block] Signature: {sig_len} chars")
                print("(Summarized thinking - signature proves reasoning occurred)\n")
            elif block.type == "text":
                text = block.text
                print(f"Response:\n{text[:1000]}..." if len(text) > 1000 else f"Response:\n{text}")

        print(
            f"\nUsage: Input={message.usage.input_tokens}, "
            f"Output={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

### Comparing effort levels

Different effort levels produce different reasoning depths and costs. Use this to find the right balance for your workload:

In [ ]:
# Compare effort levels on the same prompt
def test_effort_level(effort_level: str, prompt: str):
    if not MANTLE_AVAILABLE:
        return {"effort": effort_level, "error": "SDK not installed"}

    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=12000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": effort_level}},
            messages=[{"role": "user", "content": prompt}],
        )

        has_thinking = any(b.type == "thinking" for b in message.content)
        response_len = sum(len(b.text) for b in message.content if b.type == "text")

        return {
            "effort": effort_level,
            "thought": has_thinking,
            "input_tokens": message.usage.input_tokens,
            "output_tokens": message.usage.output_tokens,
            "response_chars": response_len,
        }
    except Exception as e:
        return {"effort": effort_level, "error": str(e)}


prompt = (
    "Prove that there are infinitely many prime numbers. "
    "Then explain why this proof technique cannot be used to prove "
    "the twin prime conjecture."
)

print("Testing effort levels on the same prompt...\n")
print(f"{'Effort':<8} {'Thought':<8} {'Input':<8} {'Output':<8} {'Response':<10}")
print("-" * 50)

for effort in ["low", "medium", "high", "max"]:
    result = test_effort_level(effort, prompt)
    if "error" not in result:
        print(
            f"{result['effort']:<8} "
            f"{str(result['thought']):<8} "
            f"{result['input_tokens']:<8} "
            f"{result['output_tokens']:<8} "
            f"{result['response_chars']:<10}"
        )
    else:
        print(f"{result['effort']:<8} Error: {result['error']}")

print("\nHigher effort = deeper reasoning, more tokens, better quality on complex tasks.")

---

## 4. Task budgets (beta)

Task budgets let you set token spending limits for entire agentic loops. This prevents runaway costs during autonomous operations while allowing Claude to complete complex multi-step tasks.

**How it works:**
- Set a maximum token budget via `output_config.task_budget`
- Claude tracks usage across the conversation/agentic loop
- When the budget is approached, Claude wraps up gracefully rather than stopping mid-task
- Ideal for autonomous agents, long-running workflows, and cost-controlled environments

**Requires beta header:** `anthropic_beta: ["task-budgets-2026-03-13"]`

In [ ]:
# Task Budget example
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=4096,
            thinking={"type": "adaptive"},
            extra_body={
                "anthropic_beta": ["task-budgets-2026-03-13"],
                "output_config": {
                    "effort": "high",
                    "task_budget": {"type": "tokens", "total": 25000},
                },
            },
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Plan a comprehensive market research project for a new "
                        "fintech startup targeting small businesses. Include "
                        "competitor analysis, market sizing, and go-to-market "
                        "strategy recommendations."
                    ),
                }
            ],
        )

        print(f"Stop reason: {message.stop_reason}")
        print(f"Output tokens used: {message.usage.output_tokens} (budget: 25,000)")

        for block in message.content:
            if block.type == "text":
                text = block.text
                print(f"\nResponse:\n{text[:1000]}..." if len(text) > 1000 else f"\nResponse:\n{text}")

    except Exception as e:
        print(f"Error: {e}")
        print("Note: Task Budgets require the beta header and may not be available in all regions.")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 5. Context compaction (beta)

Context compaction enables "infinite context" for long-running conversations and agentic tasks by automatically summarizing older context when approaching the context window limit.

**How it works:**
1. Enable compaction with `context_management.edits` containing a `compact_20260112` edit
2. Set a `trigger` threshold (default ~500K tokens) — when input exceeds this, Claude summarizes
3. The API returns a compaction content block in the response
4. Pass the compaction block back in subsequent requests to maintain continuity

**Available with:** Messages API and InvokeModel API (not Converse API)

**Requires beta header:** `anthropic_beta: ["compact-2026-01-12"]`

In [ ]:
# Context compaction - basic setup
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=1024,
            extra_body={
                "anthropic_beta": ["compact-2026-01-12"],
                "context_management": {
                    "edits": [
                        {
                            "type": "compact_20260112",
                            "trigger": {
                                "type": "input_tokens",
                                "value": 100000,  # Trigger compaction at 100K tokens
                            },
                        }
                    ]
                },
            },
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Let's start a long-running conversation about building "
                        "a complex e-commerce platform. What are the key "
                        "architectural decisions we need to make first?"
                    ),
                }
            ],
        )

        has_compaction = any(
            getattr(b, "type", None) == "compaction" for b in message.content
        )
        print(f"Compaction triggered: {has_compaction}")
        print("(Won't trigger on short conversations - designed for 100K+ token contexts)\n")

        for block in message.content:
            if block.type == "text":
                text = block.text
                print(f"{text[:800]}..." if len(text) > 800 else text)

        print(
            f"\nUsage: Input={message.usage.input_tokens}, "
            f"Output={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error: {e}")
        print("Note: Compaction requires the beta header.")
else:
    print("Skipping - install anthropic[bedrock]")

### Multi-turn conversation with compaction

In production, you pass compaction blocks back in the conversation history to maintain context across long sessions:

In [ ]:
# Multi-turn conversation with automatic compaction handling
if MANTLE_AVAILABLE:
    conversation = []

    def chat_with_compaction(user_message: str):
        """Send a message and handle compaction blocks automatically."""
        conversation.append({"role": "user", "content": user_message})

        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=4096,
            messages=conversation,
            extra_body={
                "anthropic_beta": ["compact-2026-01-12"],
                "context_management": {
                    "edits": [
                        {
                            "type": "compact_20260112",
                            "trigger": {"type": "input_tokens", "value": 100000},
                        }
                    ]
                },
            },
        )

        # Build assistant response preserving compaction blocks
        assistant_content = []
        response_text = ""
        compacted = False

        for block in message.content:
            if block.type == "text":
                assistant_content.append({"type": "text", "text": block.text})
                response_text += block.text
            elif block.type == "compaction":
                assistant_content.append(
                    {"type": "compaction", "compaction": block.compaction}
                )
                compacted = True

        conversation.append({"role": "assistant", "content": assistant_content})

        if compacted:
            print("[Compaction occurred - older context was summarized]\n")

        return response_text

    # Multi-turn demo
    try:
        print("Turn 1:")
        r1 = chat_with_compaction("Design the database schema for an e-commerce platform")
        print(f"{r1[:400]}...\n")
        print("=" * 50)

        print("\nTurn 2:")
        r2 = chat_with_compaction("Now add authentication and authorization")
        print(f"{r2[:400]}...\n")

        print(f"Conversation: {len(conversation)} messages")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 6. Long-horizon agentic patterns

Claude Opus 4.8 is designed for long-running autonomous work. It plans across stages, tracks dependencies, recovers from errors, and re-scopes around problems rather than stalling.

This section demonstrates patterns for building agentic loops with Opus 4.8, combining adaptive thinking, task budgets, and tool use.

In [ ]:
# Agentic loop pattern with tool use and error recovery
if MANTLE_AVAILABLE:
    # Define tools the agent can use
    tools = [
        {
            "name": "search_codebase",
            "description": "Search for files or code patterns in the repository",
            "input_schema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "file_pattern": {"type": "string", "description": "Glob pattern for files"},
                },
                "required": ["query"],
            },
        },
        {
            "name": "read_file",
            "description": "Read the contents of a file",
            "input_schema": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path to read"},
                },
                "required": ["path"],
            },
        },
        {
            "name": "write_file",
            "description": "Write content to a file",
            "input_schema": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path"},
                    "content": {"type": "string", "description": "File content"},
                },
                "required": ["path", "content"],
            },
        },
    ]

    def simulate_tool_result(tool_name, tool_input):
        """Simulate tool execution for demo purposes."""
        if tool_name == "search_codebase":
            return f"Found 3 matches for '{tool_input['query']}': src/auth.py, src/middleware.py, tests/test_auth.py"
        elif tool_name == "read_file":
            return f"# Contents of {tool_input['path']}\ndef authenticate(token):\n    # TODO: implement\n    pass"
        elif tool_name == "write_file":
            return f"Successfully wrote {len(tool_input.get('content', ''))} chars to {tool_input['path']}"
        return "Tool not found"

    # Run a single agentic turn with tools
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=4096,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": "high"}},
            tools=tools,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Find the authentication module in the codebase, "
                        "read it, and implement JWT token validation. "
                        "Handle expired tokens gracefully."
                    ),
                }
            ],
        )

        print(f"Stop reason: {message.stop_reason}")
        print(f"Content blocks: {len(message.content)}\n")

        for block in message.content:
            if block.type == "thinking":
                print("[Thinking occurred]")
            elif block.type == "text":
                print(f"Text: {block.text[:300]}...")
            elif block.type == "tool_use":
                print(f"Tool call: {block.name}({json.dumps(block.input)[:100]}...)")
                # In production, execute the tool and continue the loop
                result = simulate_tool_result(block.name, block.input)
                print(f"  -> {result[:100]}")

        print(f"\nUsage: Input={message.usage.input_tokens}, Output={message.usage.output_tokens}")
        print("\nIn production, continue the loop by passing tool_results back until stop_reason='end_turn'")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 7. High-resolution vision

Claude Opus 4.8 supports high-resolution image input, providing enhanced accuracy on charts, dense documents, screenshots, and complex diagrams. This is particularly useful for financial analysis (reading dense filings and charts) and cybersecurity (analyzing screenshots and logs).

In [ ]:
# Vision example
import struct, zlib

def create_sample_chart_png():
    """Create a simple 200x100 PNG with colored bars (simulates a bar chart)."""
    w, h = 200, 100
    bar_colors = [(66, 133, 244), (234, 67, 53), (251, 188, 4), (52, 168, 83)]
    bar_heights = [80, 55, 70, 90]
    bar_width, gap, start_x = 30, 15, 20
    raw = b''
    for y in range(h):
        raw += b'\x00'
        for x in range(w):
            drawn = False
            for i, (bh, color) in enumerate(zip(bar_heights, bar_colors)):
                bx = start_x + i * (bar_width + gap)
                if bx <= x < bx + bar_width and y >= h - bh:
                    raw += bytes(color)
                    drawn = True
                    break
            if not drawn:
                raw += b'\xff\xff\xff'
    def chunk(ct, d):
        c = ct + d
        return struct.pack('>I', len(d)) + c + struct.pack('>I', zlib.crc32(c) & 0xFFFFFFFF)
    ihdr = struct.pack('>IIBBBBB', w, h, 8, 2, 0, 0, 0)
    return (b'\x89PNG\r\n\x1a\n'
            + chunk(b'IHDR', ihdr)
            + chunk(b'IDAT', zlib.compress(raw))
            + chunk(b'IEND', b''))

sample_image = create_sample_chart_png()
image_b64 = base64.b64encode(sample_image).decode('utf-8')
print(f"Generated sample chart: {len(sample_image)} bytes")
print("(In production, use real high-res images to leverage Opus 4.8's enhanced vision)\n")

if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=2048,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/png",
                                "data": image_b64,
                            },
                        },
                        {
                            "type": "text",
                            "text": "Describe this chart. What type is it? What do the colors represent?",
                        },
                    ],
                }
            ],
        )

        print(f"Vision Analysis:\n{message.content[0].text}")
        print(f"\nUsage: Input={message.usage.input_tokens}, Output={message.usage.output_tokens}")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 8. Alternative APIs (Converse and InvokeModel)

For teams already using Bedrock's native APIs, Claude Opus 4.8 works with both Converse and InvokeModel. These are useful for unified multi-model pipelines or when you need Bedrock-native integration.

In [ ]:
# Converse API
try:
    response = bedrock_runtime.converse(
        modelId=RUNTIME_MODEL_ID,
        messages=[
            {
                "role": "user",
                "content": [{"text": "What are the SOLID principles in software engineering? Be concise."}],
            }
        ],
        inferenceConfig={"maxTokens": 1024},
    )

    print("Converse API response:")
    print(response["output"]["message"]["content"][0]["text"][:500])
    print(f"\nUsage: Input={response['usage']['inputTokens']}, Output={response['usage']['outputTokens']}")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# InvokeModel API
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 1024,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Explain the CAP theorem in two sentences."}
            ],
        }
    ],
}

try:
    response = bedrock_runtime.invoke_model(
        modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body)
    )
    response_body = json.loads(response["body"].read())
    print("InvokeModel API response:")
    print(response_body["content"][0]["text"])
    print(f"\nUsage: Input={response_body['usage']['input_tokens']}, Output={response_body['usage']['output_tokens']}")

except Exception as e:
    print(f"Error: {e}")

---

## 9. Next steps

- **Explore adaptive thinking** — experiment with effort levels on your workloads to find the right quality/speed/cost balance
- **Enable compaction** for long-running agents to avoid context window limits
- **Set task budgets** for autonomous workflows to control costs
- **Try high-res vision** with real screenshots, charts, and documents from your domain
- **Build agentic loops** — Opus 4.8 excels at multi-step tool use with error recovery

### Resources

- [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/)
- [Anthropic SDK for Bedrock Mantle](https://docs.anthropic.com/en/api/claude-on-amazon-bedrock)
- [Claude model documentation](https://docs.anthropic.com/en/docs/about-claude/models)